In [1]:
from datetime import datetime
# from opera_utils import disp
import geopandas as gpd
import os
import sys
from pathlib import Path
import subprocess
from transboundary_opera import displacement_tools as dt
import matplotlib.pyplot as plt
import xarray as xr
from shapely.geometry import mapping
import ultraplot as uplt
from pyproj import CRS
import asf_search as asf
import re

# %matplotlib widget
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
env_path = sys.prefix

# Construct the path to the correct proj directory
# This path structure is standard for Conda/Pixi environments on Linux/macOS
proj_lib_path = os.path.join(env_path, 'share', 'proj')

# Set the environment variable
os.environ['PROJ_LIB'] = proj_lib_path

print(f"PROJ_LIB set to: {os.environ['PROJ_LIB']}")

In [2]:
from opera_utils.geometry import stitch_geometry_layers
from opera_utils.download import L2Product
from transboundary_opera import run1_download_DISP_S1_Static, run2_prep_mintpy_opera

ERROR 1: PROJ: internal_proj_create_from_database: /home/clayc/git_repos/Transboundary_Opera/.pixi/envs/operainsar/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 5 whereas a number >= 6 is expected. It comes from another PROJ installation.


CRSError: The EPSG code is unknown. PROJ: internal_proj_create_from_database: /home/clayc/git_repos/Transboundary_Opera/.pixi/envs/operainsar/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 5 whereas a number >= 6 is expected. It comes from another PROJ installation.

In [ ]:
import dem_stitcher

In [ ]:
[tool.pixi.feature.operainsar.dependencies]
python = "3.12.*"
pyproj = ">=3.7.2,<4"
matplotlib = ">=3.10.8,<4"
gdal = ">=3.12.0,<4"

[tool.pixi.feature.operainsar.pypi-dependencies]
ipykernel = ">=7.1.0,<8"
opera-utils = { version = ">=0.25.3,<0.26", extras = ["all"] }
dolphin = ">=0.42.0,<0.43.0"
xarray = ">=2025.11.0,<2026"
zarr = ">=3.1.5,<4"
asf-search = ">=10.2.0,<11"
rioxarray = ">=0.20.0,<0.21"
geopandas = ">=1.1.1,<2"
ipyleaflet = ">=0.20.0,<0.21"
ultraplot = ">=1.66.0,<2"
rich = { version = ">=4.2.0,<15", extras = ["jupyter"] }
boto3 = ">=1.42.8,<2"
botocore = ">=1.42.8,<2"

[tool.pixi.environments]
operainsar = ["operainsar"]   
timeseries = ["timeseries"]

[tool.pixi.tasks]

[dependency-groups]
operainsar = ["netcdf4>=1.7.3,<2", "networkx>=3.6.1,<4"]

In [ ]:
ipykernel = ">=7.1.0,<8"
dolphin = ">=0.42.2,<0.43"
xarray = ">=2025.6.1,<2026"
zarr = ">=2.18.3,<3"
asf-search = ">=11.0.0,<12"
rioxarray = ">=0.19.0,<0.20"
geopandas = ">=1.1.1,<2"
ipyleaflet = ">=0.20.0,<0.21"
ultraplot = ">=1.66.0,<2"
boto3 = ">=1.42.9,<2"
botocore = ">=1.42.9,<2"
netcdf4 = ">=1.7.3,<2"
networkx = ">=3.4.2,<4"

In [ ]:
# TODO: Maybe look at making this a pixi task? Could make it easier to run?


In [ ]:
SEARCH_START = datetime(2024, 6, 1)     # next runs should start at Jan 1, 2016
SEARCH_END = datetime(2025, 1, 1)

gdf = gpd.read_file(Path(os.getcwd()).parent.parent / "raw_data/TBA_full.shp")
# frame_ids = dt.get_opera_frame_ids(Path(os.getcwd()).parent.parent / "raw_data/TBA_full.shp")     # this only gets one set of frame ids 

In [ ]:
# Initialize a list to store frame IDs for each row
all_frame_ids = []

for entry, row in gdf.iterrows():
    
    results = asf.geo_search(
        intersectsWith=row.geometry.convex_hull.wkt,
        dataset=asf.DATASET.OPERA_S1,
        processingLevel=asf.PRODUCT_TYPE.DISP_S1,
        maxResults=50  # We only need a few results to identify the Frame IDs
    )
    
    # Set for the current row
    current_frame_ids = set()
    pattern = re.compile(r'_F(\d{5})_')

    for product in results:
        filename = product.properties['fileName']
        match = pattern.search(filename)
        if match:
            frame_id_str = match.group(1) 
            current_frame_ids.add(int(frame_id_str))
    
    # Add the sorted list of frame IDs to our collection
    all_frame_ids.append(sorted(list(current_frame_ids)))

# Assign the collected frame IDs to a new column in the GeoDataFrame
gdf['frame_ids'] = all_frame_ids

unique_frame_ids = sorted({fid for ids in gdf['frame_ids'] if isinstance(ids, (list, tuple, set)) for fid in ids})
len(unique_frame_ids)

In [ ]:
unique_frame_ids = [1092,1093,1094,1095,1096,1097,3065,3066,3067,5124,5125,7091,7092,7093,14877,14878,14879,16939,16940,18905,20694,20695,20696,20697,22665,22666,22667,24724,24725,26691,26692,28478,28479,28480,28481,28482,30452,30453,30454,30455,34477,34478,34479,38241,40295,40296,40297,40298,42264,42265,44325,44326,46290,46291]

## Trying out a different approach for downloading and preparing the data:
1. run /src/transboundary_opera/run1_download_DISP_S1_Static.py to download the OPERA DISP (displacement time series) and CSLC (orbit geometry) products
2. run /src/transboundary_opera/run2_prep_mintpy_opera.py

In [ ]:
test_frame = unique_frame_ids[0]

test_staticDir = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data" / str(test_frame) / 'orbit_data'
os.makedirs(test_staticDir, exist_ok=True)

In [ ]:
from datetime import datetime as dt


In [ ]:
# Override sys.argv to simulate command line arguments
sys.argv = [
    'notebook',  # placeholder for script name
    '--frameID', str(test_frame),
    '--version', '1.1',
    '--startDate', '20160101',
    '--endDate', dt.today().strftime('%Y%m%d'),
    '--staticDir', str(test_staticDir),
    
    '--staticOnly'  # uncomment if you only want static layers
]

inps = run1_download_DISP_S1_Static.createParser()
run1_download_DISP_S1_Static.main(inps)

In [ ]:
run1_download_DISP_S1_Static(frameID = te)

In [ ]:
static_products = {}
product = L2Product.CSLC_STATIC

for frame in unique_frame_ids:
    static_products[frame] = asf.search(
        frame=frame,
        # dataset=asf.DATASET.OPERA_S1,
        processingLevel=asf.PRODUCT_TYPE.CSLC_STATIC,
    )

In [ ]:
static_products

In [ ]:
# search CLSC Static Layer files
product = L2Product.CSLC_STATIC

results = asf.search(
    frame=test_frame,
    processingLevel=product.value,
)

# results.download(path=test_staticDir, processes=5)    # downloading static layers with simultaneous downloads

In [ ]:
product.value

In [ ]:
results

## Below works and creates LOS h5 files for input into MintPy. Though, the orbit geometry is not retrieved which makes the LOS displacement decompositiong using MintPy impossible

In [ ]:
# path to mintpy bash script
script_path = Path(os.getcwd()).parent / 'create-mintpy_cc.sh'

for frame_id in unique_frame_ids:
    out_path = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data" / str(frame_id)
    os.makedirs(out_path, exist_ok=True)

    if (out_path / "mintpy").exists():
        print(f"Frame ID {frame_id} already processed. Skipping.")
        continue
    
    else:
        # 2. Define the REQUIRED environment variables for the script
        required_env = {
            "FRAME_ID": f"{frame_id}",  # Using the current frame ID from the loop
            "START": SEARCH_START.strftime("%Y-%m-%d"),
            "END": SEARCH_END.strftime("%Y-%m-%d"),
            }

        # 3. Define OPTIONAL environment variables to override the script's defaults
        optional_env = {
            "NUM_WORKERS": "8",              # Overrides default of 4
            "REFERENCE_METHOD": "HIGH_COHERENCE"    # Overrides default of BORDER. Either HIGH_COHERENCE or BORDER would be best I think.
        }
        print(f"Preparing to execute script: {script_path}")
        print(f"With required environment: {required_env}")

        # Construct the full environment: current system environment + custom variables
        full_environment = os.environ.copy()
        full_environment.update(required_env)
        full_environment.update(optional_env)

        try:
            # Execute the script directly (preferred over shell=True)
            result = subprocess.run(
                [script_path],
                check=True,  # Raise CalledProcessError for non-zero exit codes
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,  # Decode output as string
                env=full_environment,
                cwd = out_path
            )

            # Success output
            print("\nScript Execution Successful")
            print(f"Return Code: {result.returncode}")
            print("\nSTDOUT")
            print(result.stdout)
            if result.stderr:
                print("\nSTDERR (Non-fatal messages)")
                print(result.stderr)
        except subprocess.CalledProcessError as e:
            # Error handling for script failure (e.g., if a command inside the script fails)
            print(f"\nERROR: Script Failed (Exit Code {e.returncode})")
            print(f"Command run: {e.cmd}")
            print("\nSTDOUT (Captured before failure)")
            print(e.stdout)
            print("\nSTDERR (Error messages)")
            print(e.stderr)

        except FileNotFoundError:
            # Error handling for when the script file itself cannot be found
            print(f"\nCRITICAL ERROR: Script file not found or lacks execute permission at: {script_path}")